In [1]:
from spacerocks import SpaceRock
from spacerocks.time import Time
from spacerocks.observing import Observatory, Observation
from spacerocks.spice import SpiceKernel
from spacerocks.nbody import Simulation, Force
from spacerocks.orbfit import gauss, fit_orbit_lm
import numpy as np


import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

kernel = SpiceKernel()
kernel.load("/Users/kjnapier/data/spice/latest_leapseconds.tls")
kernel.load("/Users/kjnapier/data/spice/sb441-n16.bsp")
kernel.load("/Users/kjnapier/data/spice/de441_part-1.bsp")
kernel.load("/Users/kjnapier/data/spice/de441_part-2.bsp")
kernel.load("/Users/kjnapier/data/spice/earth_1962_240827_2124_combined.bpc")

Loading kernel: /Users/kjnapier/data/spice/latest_leapseconds.tls
Loading kernel: /Users/kjnapier/data/spice/sb441-n16.bsp
Loading kernel: /Users/kjnapier/data/spice/de441_part-1.bsp
Loading kernel: /Users/kjnapier/data/spice/de441_part-2.bsp
Loading kernel: /Users/kjnapier/data/spice/earth_1962_240827_2124_combined.bpc


In [2]:
w84 = Observatory.from_obscode('w84')

In [3]:
epoch = Time.now()

name = "arrokoth"
rock = SpaceRock.from_horizons(name, epoch=epoch, origin="ssb", reference_plane="J2000")
sim = Simulation.horizons(epoch, "J2000", "ssb")
sim.add(rock)

In [4]:
observations = []    
for idx in range(0, 300, 30):
    sim.integrate(epoch + idx)
    observer = w84.at(epoch + idx, reference_plane="J2000", origin="ssb")
    rock = sim.get_particle(name)
    obs = rock.observe(observer)
    observations.append(obs)

In [5]:
smear = 0.1/3600 * (np.pi / 180)

In [34]:
simulated_observations = []
for obs in observations:
    ra = obs.ra
    dec = obs.dec
    ra += np.random.normal(0, smear)
    dec += np.random.normal(0, smear)
    epoch = obs.epoch
    observer = obs.observer
    cov = [[smear**2, 0], [0, smear**2]]
    simulated_o = Observation.from_astrometry(obs.epoch, ra, dec, obs.observer)
    simulated_o.set_covariance(cov)
    simulated_observations.append(simulated_o)

rocks = gauss(simulated_observations[0], simulated_observations[5], simulated_observations[9], min_distance=1e-6)
sim = Simulation.giants(rocks[0].epoch, "J2000", "ssb")
fit_orbit_lm(simulated_observations, rocks[0], sim)

Iteration: 0, chisq: 329.9963951200223, lambda: 0.001, ndof: 14
Iteration: 1, chisq: 94.46190428087033, lambda: 0.0001, ndof: 14
Iteration: 2, chisq: 30.503492545611127, lambda: 0.00001, ndof: 14
Iteration: 3, chisq: 14.387217167361227, lambda: 0.0000010000000000000002, ndof: 14
Iteration: 4, chisq: 10.783062791607701, lambda: 0.00000010000000000000002, ndof: 14
Iteration: 5, chisq: 8.856033833345872, lambda: 0.000000010000000000000004, ndof: 14
Iteration: 6, chisq: 8.856033833345872, lambda: 0.00000010000000000000004, ndof: 14
Iteration: 7, chisq: 8.856033833345872, lambda: 0.0000010000000000000004, ndof: 14
Iteration: 8, chisq: 8.856033833345872, lambda: 0.000010000000000000004, ndof: 14
Iteration: 9, chisq: 8.856033833345872, lambda: 0.00010000000000000005, ndof: 14
Iteration: 10, chisq: 8.856033833345872, lambda: 0.0010000000000000005, ndof: 14
Iteration: 11, chisq: 8.856033833345872, lambda: 0.010000000000000005, ndof: 14
Iteration: 12, chisq: 8.856033833345872, lambda: 0.10000000

In [56]:
rocks[0].e()

0.03673634954485915

In [ ]:
# Should return just the chisq for gradient-free methods or mcmc